In [ ]:
from pathlib import Path
import os
import sys

# Display the Python environment used by this notebook.
print("Python executable:", sys.executable)

# Locate the cloned Git repository.
current = Path.cwd().resolve()
project_root = next(
    (
        folder
        for folder in [current, *current.parents]
        if (folder / ".git").exists()
    ),
    None
)

if project_root is None:
    raise RuntimeError(
        "Please open this notebook inside your local Posters repository."
    )

# Run subsequent steps from the repository root.
os.chdir(project_root)

print("Project directory:", Path.cwd())

assert Path("src/collect.py").exists(), (
    "The crawler script was not found at src/collect.py."
)

In [ ]:
%pip install lxml Pillow

In [ ]:
# Create folders for source documents and downloaded images.
raw_folder = Path("data/raw/posters_learning")

for folder_name in ["audit", "raw_html", "images"]:
    (raw_folder / folder_name).mkdir(
        parents=True,
        exist_ok=True
    )

print("Raw data directory:", raw_folder.resolve())

In [ ]:
import urllib.request
import urllib.robotparser

BASE_URL = "https://chineseposters.net"
USER_AGENT = "PosterCourseResearch/1.0"

robots_url = f"{BASE_URL}/robots.txt"

request = urllib.request.Request(
    robots_url,
    headers={"User-Agent": USER_AGENT}
)

with urllib.request.urlopen(request, timeout=30) as response:
    robots_text = response.read().decode("utf-8")

# Preserve the rules used during collection.
robots_path = raw_folder / "audit" / "robots.txt"
robots_path.write_text(robots_text, encoding="utf-8")

# Parse the access rules.
robots = urllib.robotparser.RobotFileParser()
robots.parse(robots_text.splitlines())

poster_url = f"{BASE_URL}/posters/e39-592"

allowed = robots.can_fetch(USER_AGENT, poster_url)

print("Robots file saved to:", robots_path)
print("Poster page allowed by robots.txt:", allowed)

if not allowed:
    raise RuntimeError(
        "The site's robots.txt does not allow access to this poster page."
    )

In [ ]:
import time
from datetime import datetime, timezone

# Respect the site's crawl delay, with a minimum of two seconds.
crawl_delay = robots.crawl_delay(USER_AGENT)
request_delay = max(2, crawl_delay or 0)

html_path = raw_folder / "raw_html" / "e39-592.html"

if html_path.exists():
    # Reuse the saved page instead of requesting it again.
    html_bytes = html_path.read_bytes()
    print("Loaded the cached HTML file.")
else:
    time.sleep(request_delay)

    request = urllib.request.Request(
        poster_url,
        headers={"User-Agent": USER_AGENT}
    )

    with urllib.request.urlopen(request, timeout=30) as response:
        html_bytes = response.read()

    html_path.write_bytes(html_bytes)
    print("Downloaded and saved the poster page.")

print("Source URL:", poster_url)
print("HTML file:", html_path.resolve())
print("File size in bytes:", len(html_bytes))

In [ ]:
from lxml import html as lxml_html

# Parse the saved HTML using its UTF-8 encoding.
document = lxml_html.fromstring(
    html_bytes.decode("utf-8")
)

# Locate the page title.
title_elements = document.xpath(
    '//h1[contains(@class, "page-title")]'
)

if not title_elements:
    raise ValueError(
        "The page title was not found. Inspect the saved HTML."
    )

title_en = " ".join(
    title_elements[0].text_content().split()
)

print("English title:", title_en)

In [ ]:
# Locate the poster detail section.
poster_articles = document.xpath(
    '//article[contains(@class, "node--type-posters")]'
)

if not poster_articles:
    raise ValueError(
        "The poster detail section was not found."
    )

poster_article = poster_articles[0]

print("Poster detail section found.")

In [ ]:
from urllib.parse import urljoin


def clean_text(element):
    """Extract text and normalize whitespace."""
    return " ".join(element.text_content().split())


def extract_field(scope, field_name):
    """Extract a Drupal field's text and associated links."""
    class_name = f" field--name-{field_name} "

    elements = scope.xpath(
        './/*[contains('
        'concat(" ", normalize-space(@class), " "), '
        '$class_name)]',
        class_name=class_name
    )

    if not elements:
        return "", []

    field_element = elements[0]

    items = field_element.xpath(
        './/*[contains('
        'concat(" ", normalize-space(@class), " "), '
        '" field__item ")]'
    )

    if not items:
        items = [field_element]

    value = " | ".join(clean_text(item) for item in items)

    links = [
        {
            "name": clean_text(link),
            "url": urljoin(BASE_URL, link.get("href"))
        }
        for link in field_element.xpath(".//a[@href]")
    ]

    return value, links


print("Field extraction functions are ready.")

In [ ]:
# Map output column names to the website's HTML field names.
field_mapping = {
    "title_zh": "field-titlechinese",
    "title_pinyin": "field-titlepinyin",
    "date_raw": "field-datetxt",
    "designers_raw": "field-taxonomy-artists",
    "publisher_raw": "field-publisher",
    "size_raw": "field-size",
    "call_number": "field-call-nr-",
    "collection": "field-collection",
    "notes_raw": "body",
    "themes_raw": "field-see-also-these-pages-",
    "catalog_raw": "field-iisg-identifier-2"
}

poster_id = poster_url.rstrip("/").rsplit("/", 1)[-1]

record = {
    "poster_id": poster_id,
    "source_url": poster_url,
    "title_en": title_en
}

for column_name, field_name in field_mapping.items():
    value, links = extract_field(
        poster_article,
        field_name
    )

    record[column_name] = value

    if column_name == "designers_raw":
        record["designers"] = links

    elif column_name == "themes_raw":
        record["themes_site"] = links

    elif column_name == "catalog_raw":
        record["catalog_links"] = links

print("Poster ID:", record["poster_id"])
print("Chinese title:", record["title_zh"])
print("Date:", record["date_raw"])
print("Designers:", record["designers_raw"])
print("Publisher:", record["publisher_raw"])
print("Physical size:", record["size_raw"])
print("Collection:", record["collection"])
print("Themes:", record["themes_raw"])

In [ ]:
tag_elements = document.xpath(
    '//section[contains(@id, "tag-block")]'
    '//a[starts-with(@href, "/tags/")]'
)

record["tags_site"] = [
    {
        "name": clean_text(tag),
        "url": urljoin(BASE_URL, tag.get("href"))
    }
    for tag in tag_elements
]

print("Website tags:")

for tag in record["tags_site"]:
    print("-", tag["name"])

In [ ]:
image_elements = poster_article.xpath(
    './/*[contains(@class, "field--name-field-image")]'
    '//img[@src]'
)

record["images"] = [
    {
        "image_url": urljoin(
            poster_url,
            image.get("src")
        ),
        "image_alt": image.get("alt", "")
    }
    for image in image_elements
]

if not record["images"]:
    raise ValueError(
        "No poster display image was found."
    )

print("Display images found:", len(record["images"]))

for image in record["images"]:
    print("Image URL:", image["image_url"])
    print("Alternative text:", image["image_alt"])

In [ ]:
import hashlib

record["missing_fields"] = [
    column_name
    for column_name in field_mapping
    if not record[column_name]
]

record["html_sha256"] = hashlib.sha256(
    html_bytes
).hexdigest()

# This is the extraction time, not necessarily the download time.
record["extracted_at_utc"] = datetime.now(
    timezone.utc
).isoformat()

print("Missing fields:", record["missing_fields"])
print("HTML SHA256:", record["html_sha256"])

In [ ]:
import json

print(
    json.dumps(
        record,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
import csv

json_path = raw_folder / f"{poster_id}_metadata.json"
csv_path = raw_folder / f"{poster_id}_metadata.csv"

# JSON preserves nested lists and dictionaries.
json_path.write_text(
    json.dumps(record, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

# CSV stores nested values as JSON strings.
csv_record = {
    key: (
        json.dumps(value, ensure_ascii=False)
        if isinstance(value, (list, dict))
        else value
    )
    for key, value in record.items()
}

with csv_path.open(
    "w",
    encoding="utf-8-sig",
    newline=""
) as file:
    writer = csv.DictWriter(
        file,
        fieldnames=list(csv_record)
    )
    writer.writeheader()
    writer.writerow(csv_record)

print("JSON saved to:", json_path.resolve())
print("CSV saved to:", csv_path.resolve())

In [ ]:
from urllib.parse import urlparse

for index, image_info in enumerate(record["images"], start=1):
    image_url = image_info["image_url"]

    # Check the image URL against the site's access rules.
    if not robots.can_fetch(USER_AGENT, image_url):
        raise RuntimeError(
            f"Image access is disallowed by robots.txt: {image_url}"
        )

    extension = Path(urlparse(image_url).path).suffix or ".jpg"
    suffix = "" if index == 1 else f"_{index}"

    image_path = (
        raw_folder / "images" / f"{poster_id}{suffix}{extension}"
    )

    if image_path.exists():
        image_bytes = image_path.read_bytes()
        print("Loaded cached image:", image_path.name)
    else:
        time.sleep(request_delay)

        request = urllib.request.Request(
            image_url,
            headers={"User-Agent": USER_AGENT}
        )

        with urllib.request.urlopen(request, timeout=30) as response:
            image_bytes = response.read()

        image_path.write_bytes(image_bytes)
        print("Downloaded image:", image_path.name)

    # Verify that the downloaded file is a readable image.
    with Image.open(image_path) as image:
        image.verify()

    with Image.open(image_path) as image:
        width, height = image.size

    # Store a path relative to the dataset directory.
    image_info.update({
        "image_path": image_path.relative_to(raw_folder).as_posix(),
        "image_width": width,
        "image_height": height,
        "image_bytes": len(image_bytes),
        "image_sha256": hashlib.sha256(image_bytes).hexdigest()
    })

    print("Pixel dimensions:", width, "x", height)
    print("File size in bytes:", len(image_bytes))

In [ ]:
from IPython.display import display

first_image_path = (
    raw_folder / record["images"][0]["image_path"]
)

with Image.open(first_image_path) as image:
    display(image.copy())

In [ ]:
import urllib.error
import urllib.parse

last_request_time = time.monotonic()

request_log_path = raw_folder / "audit" / "requests.jsonl"


def fetch_bytes(url, destination):
    """Download a permitted URL or reuse its local cache."""
    global last_request_time

    destination = Path(destination)

    if urllib.parse.urlparse(url).hostname != "chineseposters.net":
        raise ValueError(f"Off-site download excluded: {url}")

    if not robots.can_fetch(USER_AGENT, url):
        raise RuntimeError(f"Access disallowed by robots.txt: {url}")

    if destination.exists() and destination.stat().st_size > 0:
        return destination.read_bytes()

    destination.parent.mkdir(parents=True, exist_ok=True)

    # Space requests to reduce load on the website.
    elapsed = time.monotonic() - last_request_time
    time.sleep(max(0, request_delay - elapsed))
    last_request_time = time.monotonic()

    status = "error"

    try:
        request = urllib.request.Request(
            url,
            headers={"User-Agent": USER_AGENT}
        )

        with urllib.request.urlopen(request, timeout=45) as response:
            if urllib.parse.urlparse(response.url).hostname != "chineseposters.net":
                raise ValueError("Off-site redirect excluded.")

            status = response.status
            content = response.read()

        if not content:
            raise ValueError("The response was empty.")

        # Use a temporary file to avoid caching a partial download.
        temporary_path = destination.with_suffix(
            destination.suffix + ".tmp"
        )
        temporary_path.write_bytes(content)
        temporary_path.replace(destination)

        return content

    except urllib.error.HTTPError as error:
        status = error.code
        raise

    finally:
        log_entry = {
            "url": url,
            "requested_at_utc": datetime.now(timezone.utc).isoformat(),
            "status": status
        }

        with request_log_path.open("a", encoding="utf-8") as file:
            file.write(json.dumps(log_entry) + "\n")

In [ ]:
import io


def scrape_poster(url):
    """Extract one poster record and download its display images."""
    poster_id = url.rstrip("/").rsplit("/", 1)[-1]

    html_path = raw_folder / "raw_html" / f"{poster_id}.html"
    html_bytes = fetch_bytes(url, html_path)

    document = lxml_html.fromstring(
        html_bytes.decode("utf-8")
    )

    articles = document.xpath(
        '//article[contains(@class, "node--type-posters")]'
    )

    if not articles:
        raise ValueError("Poster detail section not found.")

    article = articles[0]

    headings = document.xpath(
        '//h1[contains(@class, "page-title")]'
    )

    result = {
        "poster_id": poster_id,
        "source_url": url,
        "title_en": clean_text(headings[0]) if headings else ""
    }

    # Extract the original website fields.
    for column_name, field_name in field_mapping.items():
        value, links = extract_field(article, field_name)
        result[column_name] = value

        if column_name == "designers_raw":
            result["designers"] = links
        elif column_name == "themes_raw":
            result["themes_site"] = links
        elif column_name == "catalog_raw":
            result["catalog_links"] = links

    # Extract the website tags.
    tags = document.xpath(
        '//section[contains(@id, "tag-block")]'
        '//a[starts-with(@href, "/tags/")]'
    )

    result["tags_site"] = [
        {
            "name": clean_text(tag),
            "url": urljoin(BASE_URL, tag.get("href"))
        }
        for tag in tags
    ]

    # Extract and download all poster display images.
    image_elements = article.xpath(
        './/*[contains(@class, "field--name-field-image")]'
        '//img[@src]'
    )

    if not image_elements:
        raise ValueError("No poster display image found.")

    result["images"] = []

    for index, element in enumerate(image_elements, start=1):
        image_url = urljoin(url, element.get("src"))
        extension = Path(urlparse(image_url).path).suffix or ".jpg"
        suffix = "" if index == 1 else f"_{index}"

        image_path = (
            raw_folder / "images" / f"{poster_id}{suffix}{extension}"
        )

        image_bytes = fetch_bytes(image_url, image_path)

        with Image.open(io.BytesIO(image_bytes)) as image:
            image.verify()

        with Image.open(io.BytesIO(image_bytes)) as image:
            width, height = image.size

        result["images"].append({
            "image_url": image_url,
            "image_alt": element.get("alt", ""),
            "image_path": image_path.relative_to(raw_folder).as_posix(),
            "image_width": width,
            "image_height": height,
            "image_bytes": len(image_bytes),
            "image_sha256": hashlib.sha256(image_bytes).hexdigest()
        })

    result["missing_fields"] = [
        column
        for column in field_mapping
        if not result[column]
    ]

    result["html_sha256"] = hashlib.sha256(html_bytes).hexdigest()
    result["extracted_at_utc"] = datetime.now(timezone.utc).isoformat()
    result["fetch_status"] = "ok"

    return result


print("The poster scraping function is ready.")

In [ ]:
catalog_url = f"{BASE_URL}/posters/posters?page=0"
catalog_path = raw_folder / "raw_html" / "catalog_0.html"

catalog_bytes = fetch_bytes(catalog_url, catalog_path)

catalog_document = lxml_html.fromstring(
    catalog_bytes.decode("utf-8")
)

relative_links = catalog_document.xpath(
    '//div[contains(@class, "view-content")]'
    '//a[starts-with(@href, "/posters/")]/@href'
)

# Remove catalogue links and duplicate detail-page URLs.
poster_urls = list(dict.fromkeys(
    urljoin(BASE_URL, link)
    for link in relative_links
    if urllib.parse.urlparse(link).path.rstrip("/") != "/posters/posters"
))

if not poster_urls:
    raise ValueError(
        "No poster links found. Inspect the saved catalogue HTML."
    )

print("Unique poster URLs on this page:", len(poster_urls))

for url in poster_urls[:5]:
    print(url)

In [ ]:
pilot_urls = poster_urls[:5]

records_folder = raw_folder / "records"
records_folder.mkdir(parents=True, exist_ok=True)

batch_records = []
batch_errors = []

for index, url in enumerate(pilot_urls, start=1):
    poster_id = url.rstrip("/").rsplit("/", 1)[-1]

    try:
        result = scrape_poster(url)
        batch_records.append(result)

        checkpoint_path = records_folder / f"{poster_id}.json"
        checkpoint_path.write_text(
            json.dumps(result, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )

        print(f"{index}/{len(pilot_urls)} OK: {poster_id}")

    except urllib.error.HTTPError as error:
        batch_errors.append({
            "source_url": url,
            "error": str(error)
        })

        print(f"{index}/{len(pilot_urls)} HTTP ERROR: {poster_id}")

        # Stop rather than repeatedly request a blocked or rate-limited site.
        if error.code in {401, 403, 429}:
            break

    except Exception as error:
        batch_errors.append({
            "source_url": url,
            "error": str(error)
        })

        print(f"{index}/{len(pilot_urls)} ERROR: {poster_id}")

print("Successful records:", len(batch_records))
print("Failed records:", len(batch_errors))

In [ ]:
# Preserve nested metadata in JSON.
(raw_folder / "metadata_pilot_5.json").write_text(
    json.dumps(batch_records, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

# Preserve failures for inspection.
(raw_folder / "audit" / "errors_pilot_5.json").write_text(
    json.dumps(batch_errors, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

if batch_records:
    columns = list(dict.fromkeys(
        key
        for row in batch_records
        for key in row
    ))

    with (raw_folder / "metadata_pilot_5.csv").open(
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as file:
        writer = csv.DictWriter(file, fieldnames=columns)
        writer.writeheader()

        for row in batch_records:
            writer.writerow({
                key: (
                    json.dumps(value, ensure_ascii=False)
                    if isinstance(value, (list, dict))
                    else value
                )
                for key, value in row.items()
            })

print("Pilot metadata and error log saved.")

In [ ]:
def discover_poster_urls(limit=100):
    """Follow catalogue pages and collect unique poster URLs."""
    catalogue_url = f"{BASE_URL}/posters/posters?page=0"

    discovered_urls = []
    seen_poster_urls = set()
    seen_catalogue_urls = set()
    page_manifest = []

    while catalogue_url:
        if catalogue_url in seen_catalogue_urls:
            raise RuntimeError("A catalogue pagination loop was detected.")

        seen_catalogue_urls.add(catalogue_url)

        query = urllib.parse.parse_qs(
            urllib.parse.urlparse(catalogue_url).query
        )
        page_number = query.get("page", ["0"])[0]

        destination = (
            raw_folder / "raw_html" / f"catalog_{page_number}.html"
        )

        content = fetch_bytes(catalogue_url, destination)
        page = lxml_html.fromstring(content.decode("utf-8"))

        links = page.xpath(
            '//div[contains(@class, "view-content")]'
            '//a[starts-with(@href, "/posters/")]/@href'
        )

        new_count = 0

        for link in links:
            full_url = urljoin(catalogue_url, link)

            if urllib.parse.urlparse(full_url).path.rstrip("/") == "/posters/posters":
                continue

            if full_url not in seen_poster_urls:
                seen_poster_urls.add(full_url)
                discovered_urls.append(full_url)
                new_count += 1

        if not links:
            raise ValueError(
                f"No catalogue links found on page {page_number}."
            )

        page_manifest.append({
            "page_url": catalogue_url,
            "new_unique_urls": new_count,
            "html_sha256": hashlib.sha256(content).hexdigest()
        })

        print(
            f"Catalogue page {page_number}: "
            f"{len(discovered_urls)} unique URLs discovered"
        )

        if limit and len(discovered_urls) >= limit:
            break

        next_links = page.xpath('//a[@rel="next"]/@href')

        catalogue_url = (
            urljoin(catalogue_url, next_links[0])
            if next_links
            else None
        )

    selected_urls = (
        discovered_urls[:limit]
        if limit
        else discovered_urls
    )

    manifest = {
        "requested_limit": limit,
        "selected_count": len(selected_urls),
        "selected_urls": selected_urls,
        "pages": page_manifest
    }

    manifest_path = (
        raw_folder / "audit" / f"catalogue_manifest_{limit}.json"
    )
    manifest_path.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    return selected_urls

In [ ]:
pilot_100_urls = discover_poster_urls(limit=100)

print("Selected poster URLs:", len(pilot_100_urls))

assert len(pilot_100_urls) == len(set(pilot_100_urls))

In [ ]:
def collect_batch(urls, batch_name):
    """Collect poster records with checkpoints and error logging."""
    records_folder = raw_folder / "records"
    records_folder.mkdir(parents=True, exist_ok=True)

    successful_records = []
    failed_records = []
    stopped_early = False

    for index, url in enumerate(urls, start=1):
        poster_id = url.rstrip("/").rsplit("/", 1)[-1]
        checkpoint = records_folder / f"{poster_id}.json"

        try:
            result = None

            if checkpoint.exists():
                saved = json.loads(
                    checkpoint.read_text(encoding="utf-8")
                )

                images_exist = (
                    bool(saved.get("images"))
                    and all(
                        (raw_folder / image["image_path"]).exists()
                        for image in saved["images"]
                    )
                )

                if saved.get("fetch_status") == "ok" and images_exist:
                    result = saved

            if result is None:
                result = scrape_poster(url)

                checkpoint.write_text(
                    json.dumps(result, ensure_ascii=False, indent=2),
                    encoding="utf-8"
                )

            successful_records.append(result)
            print(f"{index}/{len(urls)} OK: {poster_id}")

        except urllib.error.HTTPError as error:
            failed_records.append({
                "poster_id": poster_id,
                "source_url": url,
                "error": str(error)
            })

            print(f"{index}/{len(urls)} HTTP ERROR: {poster_id}")

            if error.code in {401, 403, 429}:
                stopped_early = True
                print("Collection stopped due to an access restriction.")
                break

        except Exception as error:
            failed_records.append({
                "poster_id": poster_id,
                "source_url": url,
                "error": str(error)
            })

            print(f"{index}/{len(urls)} ERROR: {poster_id}")

    # Save successful records.
    json_path = raw_folder / f"metadata_{batch_name}.json"
    json_path.write_text(
        json.dumps(successful_records, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    if successful_records:
        columns = list(dict.fromkeys(
            key
            for row in successful_records
            for key in row
        ))

        csv_path = raw_folder / f"metadata_{batch_name}.csv"

        with csv_path.open(
            "w", encoding="utf-8-sig", newline=""
        ) as file:
            writer = csv.DictWriter(file, fieldnames=columns)
            writer.writeheader()

            for row in successful_records:
                writer.writerow({
                    key: (
                        json.dumps(value, ensure_ascii=False)
                        if isinstance(value, (list, dict))
                        else value
                    )
                    for key, value in row.items()
                })

    # Keep failed and unattempted records distinguishable.
    report = {
        "selected": len(urls),
        "successful": len(successful_records),
        "failed": len(failed_records),
        "unattempted": (
            len(urls) - len(successful_records) - len(failed_records)
        ),
        "stopped_early": stopped_early,
        "errors": failed_records
    }

    report_path = raw_folder / "audit" / f"report_{batch_name}.json"
    report_path.write_text(
        json.dumps(report, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    print(json.dumps(report, ensure_ascii=False, indent=2))

    return successful_records, report

In [ ]:
records_100, report_100 = collect_batch(
    pilot_100_urls,
    batch_name="pilot_100"
)

In [ ]:
all_poster_urls = discover_poster_urls(limit=0)

print("Total unique poster URLs:", len(all_poster_urls))

In [ ]:
all_records, full_report = collect_batch(
    all_poster_urls,
    batch_name="full"
)

In [ ]:
from pathlib import Path
import json
import pandas as pd

raw_folder = Path("data/raw/posters_learning")

metadata_path = raw_folder / "metadata_full.json"
report_path = raw_folder / "audit" / "report_full.json"

assert metadata_path.exists(), "Full metadata file was not found."
assert report_path.exists(), "Full collection report was not found."

records = json.loads(
    metadata_path.read_text(encoding="utf-8")
)

collection_report = json.loads(
    report_path.read_text(encoding="utf-8")
)

print("Collection report:")
print(json.dumps(collection_report, indent=2))

poster_ids = [record["poster_id"] for record in records]

print("\nSaved records:", len(records))
print("Unique poster IDs:", len(set(poster_ids)))

assert len(poster_ids) == len(set(poster_ids)), (
    "Duplicate poster IDs found."
)

assert len(records) == collection_report["successful"], (
    "Saved record count differs from the collection report."
)

In [ ]:
import hashlib
import io
import time
from PIL import Image

image_issues = []
image_count = 0

total_images = sum(
    len(record.get("images", []))
    for record in records
)

started_at = time.monotonic()

print(
    f"Starting validation of {total_images} images...",
    flush=True
)

for record in records:
    images = record.get("images", [])

    if not images:
        image_issues.append({
            "poster_id": record["poster_id"],
            "issue": "No image metadata"
        })
        continue

    for image_info in images:
        image_count += 1

        try:
            image_path = raw_folder / image_info["image_path"]

            # Read the file once.
            content = image_path.read_bytes()

            if len(content) != image_info["image_bytes"]:
                raise ValueError("File size mismatch")

            checksum = hashlib.sha256(content).hexdigest()

            if checksum != image_info["image_sha256"]:
                raise ValueError("SHA256 mismatch")

            # Inspect the image using the bytes already in memory.
            with Image.open(io.BytesIO(content)) as image:
                expected_size = (
                    image_info["image_width"],
                    image_info["image_height"]
                )

                if image.size != expected_size:
                    raise ValueError("Pixel dimensions mismatch")

                image.verify()

        except Exception as error:
            image_issues.append({
                "poster_id": record["poster_id"],
                "image_path": image_info.get("image_path", ""),
                "issue": str(error)
            })

        if image_count % 100 == 0 or image_count == total_images:
            elapsed = time.monotonic() - started_at

            print(
                f"Checked {image_count}/{total_images} images | "
                f"Issues: {len(image_issues)} | "
                f"Elapsed: {elapsed:.1f} seconds",
                flush=True
            )

print("Validation finished.", flush=True)
display(pd.DataFrame(image_issues).head(10))

In [ ]:
fields_to_check = [
    "title_zh",
    "date_raw",
    "designers_raw",
    "publisher_raw",
    "size_raw",
    "notes_raw",
    "themes_site",
    "tags_site"
]

missing_summary = pd.DataFrame([
    {
        "field": field,
        "missing_count": sum(
            not record.get(field)
            for record in records
        ),
        "total_records": len(records)
    }
    for field in fields_to_check
])

missing_summary["missing_percent"] = (
    missing_summary["missing_count"]
    / missing_summary["total_records"]
    * 100
)

display(missing_summary)

In [ ]:
validation_report = {
    "saved_records": len(records),
    "unique_poster_ids": len(set(poster_ids)),
    "image_records_checked": image_count,
    "image_issue_count": len(image_issues),
    "image_issues": image_issues
}

(raw_folder / "audit" / "validation_full.json").write_text(
    json.dumps(validation_report, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

missing_summary.to_csv(
    raw_folder / "audit" / "missing_fields_full.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Full collection validation saved.")